# Giọng clone cho Luyện Luật (F5-TTS tiếng Việt)

Sinh mp3 cho **mọi câu web có thể đọc** bằng giọng mẫu đã được phép dùng, rồi đưa lên web qua workflow **Nhập giọng clone**.

- Model: [hynt/F5-TTS-Vietnamese-ViVoice](https://huggingface.co/hynt/F5-TTS-Vietnamese-ViVoice) — giấy phép **CC BY-NC-SA 4.0: chỉ dùng phi thương mại**. Mã: [nguyenthienhy/F5-TTS-Vietnamese](https://github.com/nguyenthienhy/F5-TTS-Vietnamese).
- Cần GPU: **Runtime → Change runtime type → T4 GPU**. ~13 000 câu, ước 4–6 giờ GPU. Kết quả lưu vào Google Drive sau MỖI câu → mất kết nối thì chạy lại từ ô 1, tự làm tiếp phần còn thiếu (có thể chia nhiều ngày; Kaggle cũng chạy được).
- Giọng mẫu: 8–10 giây, đã tách nhạc nền, kèm **đúng lời đọc** của đoạn đó.

**Thứ tự:** chạy lần lượt ô 1 → 5, nghe thử ở ô 5; ưng thì chạy ô 6 (lâu) rồi ô 7.

In [ ]:
# 1. Kiểm tra GPU, gắn Google Drive (nơi lưu kết quả)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/luyen-luat-tts/clone'      # mp3 từng câu
ZIP = '/content/drive/MyDrive/luyen-luat-tts/tts-clone.zip'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
# 2. Cài F5-TTS tiếng Việt + tải model (~1,3 GB). Nếu báo lỗi numpy: Runtime → Restart session rồi chạy lại từ ô 1.
!pip -q install git+https://github.com/nguyenthienhy/F5-TTS-Vietnamese huggingface_hub
!apt-get -qq install -y ffmpeg > /dev/null
from huggingface_hub import hf_hub_download
CKPT = hf_hub_download('hynt/F5-TTS-Vietnamese-ViVoice', 'model_last.pt')
VOCAB = hf_hub_download('hynt/F5-TTS-Vietnamese-ViVoice', 'config.json')  # tệp này thực chất là bảng từ vựng (vocab)
print(CKPT, VOCAB)

In [ ]:
# 3. Lấy mã web + gom mọi câu web có thể đọc (dùng chính code app: npm run tts:texts)
%cd /content
!rm -rf Law && git clone -q --depth 1 https://github.com/tranquanghaofake-dotcom/Law
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get -qq install -y nodejs > /dev/null
%cd /content/Law
!npm ci --no-audit --no-fund --silent && npm run content --silent && TTS_TEXTS=/content/texts.json npm run tts:texts -w @luyen-luat/web --silent
%cd /content
import sys, json
sys.path.insert(0, '/content/Law/tools/tts')
from gen import text_key, collect, norm, TRIM   # cùng khoá tệp với app (FNV-1a của câu gốc)
from vi_norm import normalize                   # đọc số, chữ viết tắt thành chữ
texts, seen = [], set()
for s in ['Chính xác.', 'Chưa đúng.'] + collect() + json.load(open('/content/texts.json')):
    s = norm(s)
    if s and text_key(s) not in seen:
        seen.add(text_key(s)); texts.append(s)
# câu ngắn/hay gặp trước: phủ được nhiều lời đọc sớm nếu phải dừng giữa chừng
texts = texts[:2] + sorted(texts[2:], key=len)
print(len(texts), 'câu')

In [ ]:
# 4. Giọng mẫu: tải lên tệp ref.wav (8–10 giây, đã tách nhạc nền) và ghi ĐÚNG lời đọc trong đó (chữ thường).
from google.colab import files
up = files.upload()
REF = '/content/' + list(up)[0]
REF_TEXT = ('bạn không thể nói với pháp luật rằng đây là người lương thiện hay bảo vệ họ, đây là kẻ ác hay kết tội họ. '
            'kẻ ác hay người lương thiện chưa bao giờ là đối tượng thuộc diện bảo vệ của pháp luật.')
VOICE = 'clone:kienthucxahoi0'   # tên bộ giọng ghi vào manifest
STYLE = 'f5-vivoice-v1'          # đổi khi đổi giọng mẫu/thông số → bộ mới
SPEED = 1.0                      # <1 chậm hơn, >1 nhanh hơn
NFE = 32                         # bước khử nhiễu: 32 cân bằng; 16 nhanh gấp đôi nhưng kém hơn

In [ ]:
# 5. Nạp model + NGHE THỬ 3 câu trước khi chạy hết
import subprocess
from IPython.display import Audio, display
from f5_tts.api import F5TTS
tts = F5TTS(model='F5TTS_Base', ckpt_file=CKPT, vocab_file=VOCAB)

def synth(sentence, mp3):
    tts.infer(REF, REF_TEXT, normalize(sentence), file_wave='/content/_tmp.wav', speed=SPEED, nfe_step=NFE, show_info=lambda *a, **k: None)
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', '/content/_tmp.wav', '-af', TRIM, '-ac', '1', '-ar', '24000', '-b:a', '32k', mp3], check=True)

for s in ['Chính xác.', 'Chưa đúng. Đáp án: Quốc hội.', 'Khoản 1 Điều 119 Hiến pháp 2013: Hiến pháp là luật cơ bản của nước Cộng hòa xã hội chủ nghĩa Việt Nam, có hiệu lực pháp lý cao nhất.']:
    print(s, '→', normalize(s))
    synth(s, '/content/_thu.mp3'); display(Audio('/content/_thu.mp3'))

In [ ]:
# 6. Sinh toàn bộ (lâu — có thể chạy lại bất cứ lúc nào, chỉ làm câu còn thiếu)
import os, time
from tqdm.auto import tqdm
todo = [s for s in texts if not os.path.exists(f'{OUT}/{text_key(s)}.mp3')]
print(f'còn {len(todo)}/{len(texts)} câu')
fails = []
for s in tqdm(todo):
    try:
        synth(s, f'{OUT}/{text_key(s)}.mp3')
    except Exception as e:
        fails.append((s, repr(e)))
print('lỗi:', len(fails)); print(fails[:5])

In [ ]:
# 7. Đóng gói: manifest (chỉ câu đã có tệp) + zip vào Drive → tải lên GitHub Release
import json, os, zipfile
keys = sorted(k for k in {text_key(s) for s in texts} if os.path.exists(f'{OUT}/{k}.mp3'))
json.dump({'voice': VOICE, 'style': STYLE, 'keys': keys}, open(f'{OUT}/manifest.json', 'w', encoding='utf-8'))
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_STORED) as z:
    z.write(f'{OUT}/manifest.json', 'manifest.json')
    for k in keys: z.write(f'{OUT}/{k}.mp3', f'{k}.mp3')
print(f'{len(keys)}/{len(texts)} câu → {ZIP} ({os.path.getsize(ZIP)/1e6:.0f} MB)')

## Đưa lên web
1. Tải `tts-clone.zip` từ Google Drive (thư mục `luyen-luat-tts`).
2. GitHub → repo **Law** → **Releases** → *Draft a new release* → tag **`giong-clone`** → kéo thả `tts-clone.zip` → *Publish*.
3. **Actions → Nhập giọng clone → Run workflow** (nhánh `main`). Workflow kiểm tra tệp, lưu vào cache giọng đọc rồi tự chạy **Deploy**.
4. Tải lại web 1–2 lần. Câu chưa có mp3 (thêm nội dung mới sau này) đọc bằng giọng trình duyệt — chạy lại notebook (ô 1→7, chỉ sinh câu thiếu) rồi lặp bước 2–3 (sửa Release, thay tệp zip).

Workflow **Giọng đọc** hằng tuần (edge-tts) tự bỏ qua khi cache đang là giọng clone.